# Product Confirmation Workflow

This notebook downloads DIST-ALERT products from S3, unzips them, and runs the confirmation workflow.

In [1]:
import pandas as pd
import shutil
import zipfile
import requests
from pathlib import Path
from tqdm import tqdm
import concurrent.futures
from dist_s1 import run_sequential_confirmation_of_dist_products_workflow

In [2]:
tmp_dir =  Path('tmp')
unconfirmed_products_dir =  Path('unconfirmed_products')
confirmed_products_dir =  Path('confirmed_products')

tmp_dir.mkdir(exist_ok=True)
unconfirmed_products_dir.mkdir(exist_ok=True)
confirmed_products_dir.mkdir(exist_ok=True)

In [3]:
# Load the test products CSV
csv_path = Path('amy_val_one_of_each__2025_09_15.csv')
df = pd.read_csv(csv_path)
df.head()

,name,zip_url,browse_url,product_request_time,processing_duration,high_confidence_alert_threshold,mgrs_tile_id,post_date_buffer_days,stride_for_norm_param_estimation,n_workers_for_norm_param_estimation,...,model_source,memory_strategy,batch_size_for_norm_param_estimation,post_date,track_number,low_confidence_alert_threshold,n_workers_for_despeckling,device,max_pre_imgs_per_burst_mw,model_compilation
0,dry_conditions__24MXV,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-09-15T16:15:29+00:00,1437.241,4.5,24MXV,1,7,4,...,transformer_optimized,high,32,2024-07-02,82,2.5,4,best,none,False
1,dry_conditions__24MXV,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-09-15T16:15:29+00:00,1430.626,4.5,24MXV,1,7,4,...,transformer_optimized,high,32,2024-07-14,82,2.5,4,best,none,False
2,dry_conditions__24MXV,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-09-15T16:15:29+00:00,1267.353,4.5,24MXV,1,7,4,...,transformer_optimized,high,32,2024-07-26,82,2.5,4,best,none,False
3,dry_conditions__24MXV,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-09-15T16:15:29+00:00,1447.821,4.5,24MXV,1,7,4,...,transformer_optimized,high,32,2024-08-19,82,2.5,4,best,none,False
4,dry_conditions__24MXV,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-09-15T16:15:29+00:00,1472.553,4.5,24MXV,1,7,4,...,transformer_optimized,high,32,2024-08-31,82,2.5,4,best,none,False


# Download

In [4]:
def create_download_session(max_workers: int = 5) -> requests.Session:
    """Create a requests session with appropriate settings for downloads.

    Args:
        max_workers: Number of concurrent download threads (used to size connection pool)
    """
    session = requests.Session()
    session.headers.update({'User-Agent': 'dist-s1-calval/1.0'})

    pool_maxsize = max(max_workers * 2, 10)
    pool_maxsize = min(pool_maxsize, 50)

    adapter = requests.adapters.HTTPAdapter(
        pool_connections=10,
        pool_maxsize=pool_maxsize,
        max_retries=4, 
    )
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def download_file(url, destination_path, session):
    with session.get(url, stream=True) as response:
        response.raise_for_status()
    
        with open(destination_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)
    
    return destination_path

In [5]:
records = df.to_dict(orient='records')

download_data = [{'url':r['zip_url'],
                  'dest_path': tmp_dir/ f'{Path(r['zip_url']).name}'} for r in records]
download_data[:3]

[{'url': 'https://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa.s3.us-west-2.amazonaws.com/ddaeebf0-d932-4ace-a29d-f028a966ac8f/OPERA_L3_DIST-ALERT-S1_T24MXV_20240702T080946Z_20250915T174446Z_S1_30_v0.1.zip',
  'dest_path': PosixPath('tmp/OPERA_L3_DIST-ALERT-S1_T24MXV_20240702T080946Z_20250915T174446Z_S1_30_v0.1.zip')},
 {'url': 'https://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa.s3.us-west-2.amazonaws.com/74c36dad-ed0f-43d7-80d1-5ba93093a351/OPERA_L3_DIST-ALERT-S1_T24MXV_20240714T080946Z_20250915T171855Z_S1_30_v0.1.zip',
  'dest_path': PosixPath('tmp/OPERA_L3_DIST-ALERT-S1_T24MXV_20240714T080946Z_20250915T171855Z_S1_30_v0.1.zip')},
 {'url': 'https://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa.s3.us-west-2.amazonaws.com/24464633-2210-4a6c-93b7-f41f9f0038c2/OPERA_L3_DIST-ALERT-S1_T24MXV_20240726T080945Z_20250915T175220Z_S1_30_v0.1.zip',
  'dest_path': PosixPath('tmp/OPERA_L3_DIST-ALERT-S1_T24MXV_20240726T080945Z_20250915T175220Z_S1_30_v0.1.zip')}]

In [ ]:
N_WORKERS = 10

session = create_download_session(N_WORKERS)
def download_one_with_session(input_data) -> Path:
    url = input_data['url']
    dest_path = input_data['dest_path']
    dest_path.parent.mkdir(exist_ok=True, parents=True)
    return download_file(url, dest_path, session)
with concurrent.futures.ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    zip_paths = list(tqdm(executor.map(download_one_with_session, download_data[:]), total=len(download_data[:])))

  0%|                                                                    | 0/701 [00:00<?, ?it/s]

# Unzip

In [ ]:
def unzip_file(zip_path, extract_to):
    zip_path = Path(zip_path)
    extract_to = Path(extract_to)
    
    subdirectory_name = zip_path.stem
    
    full_extract_path = extract_to / subdirectory_name
    full_extract_path.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(full_extract_path)
    
    return full_extract_path
    
for zip_path, record in zip(zip_paths, records):
    # mgrs_tile_id = downloaded_files[0].name.split('_')[3][1:]
    job_name = record['name']
    unconfirmed_products_dir = Path(f'unconfirmed_products/{job_name}')
    unconfirmed_products_dir.mkdir(exist_ok=True)
    unzip_file(zip_path, unconfirmed_products_dir)

In [10]:
ts_directories = list(Path('unconfirmed_products').glob('*/'))
ts_unzipped = [subdir.name for subdir in ts_directories]
ts_unzipped[:3]

['road_expansion__50SQE', 'shifting_cultivation__48QXD', 'tornado__16SGG']

In [11]:
# Run the confirmation workflow
for ts_dir in ts_directories:
    print(ts_dir.name)
    run_sequential_confirmation_of_dist_products_workflow(
        ts_dir, 
        confirmed_products_dir / ts_dir.name
    )

road_expansion__50SQE


Confirming 61 products:   0%|          | 0/61 [00:00<?, ?it/s]

shifting_cultivation__48QXD


Confirming 91 products:   0%|          | 0/91 [00:00<?, ?it/s]

tornado__16SGG


...

Confirming 31 products:   0%|          | 0/31 [00:00<?, ?it/s]

mining__21MWP


Confirming 66 products:   0%|          | 0/66 [00:00<?, ?it/s]

landslide__18LUR


Confirming 116 products:   0%|          | 0/116 [00:00<?, ?it/s]

In [12]:
# cleanup_temp = True
# if cleanup_temp:
#     shutil.rmtree(tmp_dir)